In [0]:
raw_path = "abfss://raw@storageaccountspotifyuk.dfs.core.windows.net/chunks"
bronze_path = "abfss://bronze@storageaccountspotifyuk.dfs.core.windows.net/hospitals"
schema_location = "abfss://bronze@storageaccountspotifyuk.dfs.core.windows.net/schema/hospitals"
check_point = "abfss://bronze@storageaccountspotifyuk.dfs.core.windows.net/checkpoints/hospitals"

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

In [0]:
bronze_df = (
    spark.readStream.
    format("cloudFiles").
    option("cloudFiles.format", "csv").
    option("header", "true").
    option("cloudFiles.schemaLocation", schema_location).
    option("cloudFiles.inferColumnTypes", "true").
    load(raw_path)
)

In [0]:
bronze_df = (
    bronze_df.withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
bronze_df.writeStream.format("delta")\
    .option("checkpointLocation", check_point)\
    .trigger(once=True)\
    .start(bronze_path)


# To Verify data loaded correctly

In [0]:
df = spark.read.format("csv").option("header", True).load(raw_path)
df.count()

In [0]:
df_from_bronze = spark.read.format("delta").load(bronze_path)
df_from_bronze.count()